# E049 — Image TTA Expansion: More Views

**Motivation:** E043 uses 5 TTA views (flip + rot -5/0/+5). Can we improve with more rotation views or additional augmentations?

**Hypothesis:** More TTA views (within distribution) will improve image EER.

**Configs:**
- `E043`: 5 views (flip + rot -5/0/+5) — baseline
- `rot7`: 7 views (flip + rot -7/-3.5/0/+3.5/+7)
- `rot9`: 9 views (flip + rot -8/-6/-4/-2/0/+2/+4/+6/+8)
- `bright`: 5 views + brightness variations
- `noise`: 5 views + noise variations

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from data.splits import load_manifest, iter_folds
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
print(f'{len(manifest)} samples')

E043_REF = {'mean_eer': 0.74, 'std_eer': 0.57}

In [ ]:
def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(stem, data_dir):
    p = _find_png(stem, data_dir)
    img = Image.open(p).convert('L')
    return np.array(img.resize((80, 80)), dtype=np.float32) / 255.0

def _aug_image_train(img, rng):
    """Training augmentation (adversarial + flip + brightness + noise)."""
    # Flip
    if rng.random() < 0.5:
        img = np.fliplr(img)
    # Brightness
    factor = rng.uniform(0.7, 1.3)
    img = np.clip(img * factor, 0, 1)
    # Noise
    noise = rng.normal(0, 0.05, img.shape)
    img = np.clip(img + noise, 0, 1)
    # Adversarial rotation
    angle = rng.uniform(-10, 10)
    img = rotate(img, angle, reshape=False, mode='reflect')
    return img

def train_image_model(train_df, data_dir, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in train_df.iterrows():
        img = _load_image(row["stem"], data_dir)
        if row["label"] == 1:
            img_aug = _aug_image_train(img, rng)
            X.append(img.flatten())
            X.append(img_aug.flatten())
            y.extend([1, 1])
        else:
            X.append(img.flatten())
            y.append(0)
    
    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_pca = pca.fit_transform(X)
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(X_pca, y)
    return pca, clf

def score_image_tta(pca, clf, test_row, data_dir, tta_type='E043'):
    img = _load_image(test_row["stem"], data_dir)
    scores = []
    
    if tta_type == 'E043':
        # 5 views: flip + rot -5/0/+5
        imgs = [img, np.fliplr(img)]
        for angle in [-5, 0, 5]:
            if angle != 0:
                imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
    
    elif tta_type == 'rot7':
        # 7 views: flip + rot -7/-3.5/0/+3.5/+7
        imgs = [img, np.fliplr(img)]
        for angle in [-7, -3.5, 0, 3.5, 7]:
            if angle != 0:
                imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
    
    elif tta_type == 'rot9':
        # 9 views: flip + rot -8/-6/-4/-2/0/+2/+4/+6/+8
        imgs = [img, np.fliplr(img)]
        for angle in [-8, -6, -4, -2, 0, 2, 4, 6, 8]:
            if angle != 0:
                imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
    
    elif tta_type == 'bright':
        # 5 views + brightness
        imgs = [img, np.fliplr(img)]
        for angle in [-5, 0, 5]:
            if angle != 0:
                imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
        # Add brightness variations
        for factor in [0.9, 1.1]:
            imgs.append(np.clip(img * factor, 0, 1))
    
    elif tta_type == 'noise':
        # 5 views + noise
        imgs = [img, np.fliplr(img)]
        for angle in [-5, 0, 5]:
            if angle != 0:
                imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
        # Add noise variations
        rng = np.random.default_rng(SEED)
        for std in [0.03, 0.05]:
            imgs.append(np.clip(img + rng.normal(0, std, img.shape), 0, 1))
    
    else:
        raise ValueError(f'Unknown TTA type: {tta_type}')
    
    for img_view in imgs:
        x = img_view.flatten().reshape(1, -1)
        x_pca = pca.transform(x)
        log_odds = clf.decision_function(x_pca)[0]
        scores.append(log_odds)
    
    return np.mean(scores)

print('Functions defined')

In [ ]:
tta_configs = ['E043', 'rot7', 'rot9', 'bright', 'noise']
results = []

for tta_type in tta_configs:
    print(f'\n{"="*60}')
    print(f'Testing TTA: {tta_type}')
    print(f'{"="*60}')
    
    fold_eers = []
    
    for fold_idx, tr_idx, te_idx in iter_folds(manifest, n_splits=3, seed=SEED):
        print(f'  Fold {fold_idx}...', end=' ', flush=True)
        
        train_df = manifest.iloc[tr_idx]
        test_df = manifest.iloc[te_idx]
        
        pca, clf = train_image_model(train_df, DATA, SEED + fold_idx)
        
        scores, labels = [], []
        for _, row in test_df.iterrows():
            score = score_image_tta(pca, clf, row, DATA, tta_type=tta_type)
            scores.append(score)
            labels.append(row['label'])
        
        eer, _ = compute_eer(np.array(labels), np.array(scores))
        fold_eers.append(eer * 100)
        print(f'{eer*100:.2f}%')
    
    mean_eer = np.mean(fold_eers)
    std_eer = np.std(fold_eers)
    print(f'{tta_type}: {mean_eer:.2f} ± {std_eer:.2f}%')
    
    results.append({
        'tta_type': tta_type,
        'eer_mean': mean_eer,
        'eer_std': std_eer,
        'fold_eers': fold_eers
    })

# Summary
print('\n' + '='*70)
print('E049 IMAGE TTA EXPANSION SUMMARY')
print('='*70)

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df[['tta_type', 'eer_mean', 'eer_std']].to_string(index=False))

best_idx = results_df['eer_mean'].argmin()
best = results_df.iloc[best_idx]
print(f'\n✓ Best: {best["tta_type"]} → {best["eer_mean"]:.2f} ± {best["eer_std"]:.2f}%')

improvement = E043_REF['mean_eer'] - best['eer_mean']
print(f'Improvement vs E043 (0.74%): {improvement:+.2f}pp')

if improvement > 0.1:
    print(f'\n✓✓ NEW IMAGE FLAGSHIP!')
elif improvement > 0:
    print(f'\n✓ Marginal improvement')
else:
    print(f'\n✗ E043 remains flagship')